# 1: Group patients by similar features

## Plain English summary

This notebook uses the trained machine learning model to predict the probability of thrombolysis for each patient in the test data. This means we can save a data file with each patient, their group number, and their probability of thrombolysis.

## Load imports

In [10]:
import pandas as pd
import numpy as np
import yaml
import pickle
import copy

from dataclasses import dataclass


import sys
sys.path.append('../../../')  # path to dir containing stroke_utilities
import stroke_utilities.process_data as process_data

import matplotlib.pyplot as plt

# Turn warnings off to keep notebook tidy
import warnings
warnings.filterwarnings("ignore")

## Set up paths and filenames

In [18]:
@dataclass(frozen=True)
class Paths:
    '''Singleton object for storing paths to data and database.'''

    data_read_path: str = './stroke_utilities/data/'
    # data_read_filename: str = 'reformatted_data_thrombolysis_decision.csv'
    # data_test_filename: str = 'cohort_10000_test.csv'
    # data_train_filename: str = 'cohort_10000_train.csv'
    data_save_path: str = '.'  # './stroke_utilities/data'
    model_folder = '../../../stroke_utilities/models'
    notebook: str = ''

paths = Paths()

# Load data

Load in group assignments for all patients:

In [24]:
data = pd.read_csv('all_data_group_assignments.csv')

In [25]:
data.columns

Index(['stroke_team_id', 'stroke_severity', 'prior_disability', 'age',
       'infarction', 'onset_to_arrival_time', 'precise_onset_known',
       'onset_during_sleep', 'arrival_to_scan_time', 'afib_anticoagulant',
       'thrombolysis', 'id', 'onset_to_thrombolysis', 'any_afib_diagnosis',
       'discharge_disability', 'thrombectomy', 'data_source', 'mask_number'],
      dtype='object')

Prepare for model:

In [26]:
features_to_use = [
    'stroke_team_id',
    'stroke_severity',
    'prior_disability',
    'age',
    'infarction',
    'onset_to_arrival_time',
    'precise_onset_known',
    'onset_during_sleep',
    'arrival_to_scan_time',
    'afib_anticoagulant',
    # 'year',
    # 'thrombolysis'
]

X = data[features_to_use]

In [27]:
X = process_data.one_hot_encode_column(
    X, 'stroke_team_id', prefix='team')

Import the trained machine learning model:

In [28]:
with open(f'{paths.model_folder}/model.p', 'rb') as fp:
    model = pickle.load(fp)

## Predict probabilities of thrombolysis

In [53]:
# Only store them to the nearest %, i.e. round to 2 d.p.
probs = np.round(model.predict_proba(X)[:,1], 4)

In [54]:
data['predicted_probs'] = probs

## Save results

In [55]:
results = data[['mask_number', 'predicted_probs', 'thrombolysis', 'data_source']]

In [56]:
results.to_csv('all_data_masks_probabilities.csv', index=False)

## Average probabilities over groups

In [57]:
df_mask_numbers = pd.read_csv('mask_numbers.csv')

all_mask_numbers = df_mask_numbers['mask_number'].values

In [58]:
lists_of_probs = []

for data_source in ['all', 'train', 'test']:
    # Limit to just this data source:
    if data_source == 'all':
        df_here = results
    else:
        df_here = results[results['data_source'] == data_source]
    df_here = df_here.drop('data_source', axis='columns')
    n_here = df_here['mask_number'].value_counts()
    n_here.name = 'number_of_patients'
    df_here = df_here.groupby('mask_number').mean()
    df_here = pd.merge(df_here, n_here, left_index=True, right_index=True, how='left')
    rename_dict = dict([(c, f'{c}_{data_source}') for c in df_here.columns])
    df_here = df_here.rename(columns=rename_dict)
    lists_of_probs.append(df_here)

In [59]:
df_probs = pd.concat(lists_of_probs, axis='columns')

# Fill in missing groups:
missing_groups = list(set(all_mask_numbers) - set(df_probs.index))
for m in missing_groups:
    df_probs.loc[m] = np.NaN
df_probs = df_probs.sort_index()
df_probs = np.round(df_probs, 4)

In [60]:
df_probs

,predicted_probs_all,thrombolysis_all,number_of_patients_all,predicted_probs_train,thrombolysis_train,number_of_patients_train,predicted_probs_test,thrombolysis_test,number_of_patients_test
mask_number,,,,,,,,,
0,0.0000,0.0,1080.0,0.0000,0.0,495.0,0.0000,0.0,45.0
1,0.0000,0.0,742.0,0.0000,0.0,335.0,0.0000,0.0,36.0
2,0.0000,0.0,108.0,0.0000,0.0,53.0,0.0000,0.0,1.0
3,0.0000,0.0,82.0,0.0000,0.0,37.0,0.0000,0.0,4.0
4,0.0001,0.0,2808.0,0.0001,0.0,1271.0,0.0001,0.0,133.0
...,...,...,...,...,...,...,...,...,...
571,0.0072,0.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN
572,0.0393,0.0,15.0,NaN,NaN,NaN,NaN,NaN,NaN
573,0.0134,0.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN


In [61]:
df_probs.to_csv('group_average_predicted_probs.csv')